In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.loader import load_league_season
wnba_25 = load_league_season("wnba", "2025")

Loaded: advanced.csv
Shape: (234, 25)
Loaded: playbyplay.csv
Shape: (235, 21)
Loaded: perposs.csv
Shape: (234, 28)
Loaded: shooting.csv
Shape: (235, 25)


In [3]:
wnba_25.keys()

dict_keys(['advanced', 'playbyplay', 'perposs', 'shooting'])

In [4]:
from src.cleaner import clean_stat_table

advanced = clean_stat_table(wnba_25["advanced"])
perposs = clean_stat_table(wnba_25["perposs"])
shooting = clean_stat_table(wnba_25["shooting"])
playbyplay = clean_stat_table(wnba_25["playbyplay"])

In [6]:
from importlib import reload
import src.profiles

reload(src.profiles)

from src.profiles import build_player_profiles

profiles = build_player_profiles(
    advanced=advanced,
    perposs=perposs,
    shooting=shooting,
    playbyplay=playbyplay
)

In [7]:
from src.profile_cleanup import clean_profiles

profiles_clean = clean_profiles(profiles)

profiles_clean.shape

(234, 84)

In [8]:
from models.model import build_model_dataset
from src.database import save_df_to_sqlite
from src.profile_cleanup import clean_profiles

model_df = build_model_dataset(
    profiles_clean,
    min_minutes=500
)

model_df.shape
model_df.columns.tolist()

['player',
 'tm',
 'pos',
 'g',
 'gs',
 'mp',
 'pts',
 'ts_pct',
 'efg_pct',
 'fg_pct',
 '2p_pct',
 '3p_pct',
 'ft_pct',
 'three_point_attempt_rate',
 'free_throw_rate',
 'dist',
 '0_3',
 '16_3p',
 '0_3_2',
 '16_3p_2',
 'ast',
 'assist_pct',
 'tov',
 'turnover_pct',
 'ows',
 'ortg',
 'orb',
 'trb',
 'off_reb_pct',
 'total_reb_pct',
 'stl',
 'blk',
 'steal_pct',
 'block_pct',
 'pf',
 'drtg',
 'dws',
 'per',
 'ws',
 'on_off']

In [13]:
from src.normalize import normalize_by_league

model_df["league"] = "WNBA"
model_df["season"] = "2025"

wnba_normalized = normalize_by_league(model_df)

In [14]:
from pathlib import Path

processed_dir = Path("../DATA/processed/wnba")
processed_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
profiles.to_csv(
    processed_dir / "wnba_profiles_raw.csv",
    index=False
)

profiles_clean.to_csv(
    processed_dir / "wnba_profiles_master.csv",
    index=False
)

model_df.to_csv(
    processed_dir / "wnba_model_dataset.csv",
    index=False
)
wnba_normalized.to_csv(
    processed_dir / "wnba_model_normalized.csv",
    index=False
)

print("Saved all WNBA datasets.")

In [18]:
from src.database import save_df_to_sqlite

save_df_to_sqlite(
    model_df,
    "../DATA/database/parallel_hoops__wnba_v2.db",
    "wnba_model_dataset_25"
)

Saved 111 rows to ../DATA/database/parallel_hoops__wnba_v2.db table: wnba_model_dataset_25
